# 🚗 VLA 자율주행 — 카메라 → 웨이포인트 예측 데모 (무료 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKKUAutoLab/VLA_Simulation/blob/main/colab/VLA_Simulation_Colab.ipynb)

**이 데모가 보여주는 것**: 학습된 VLA 모델이 자동차 앞 카메라 사진 한 장을 보고,
**어디로 가야 하는지(웨이포인트 경로)** 와 **핸들을 얼마나 꺾을지(조향)** 를 예측합니다.
이것이 VLA(Vision→Action)의 핵심 원리입니다.

### 딱 2단계면 끝
1. 위 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** → 저장
2. **런타임 → 모두 실행** (`Ctrl+F9`)

> 무거운 ROS2·Gazebo 설치 없이, VLA 모델 추론만 실행하므로 Colab에서 안정적으로 돌아갑니다.
> 실시간 Gazebo 주행 화면이 필요하면 로컬(GPU 노트북) 또는 The Construct를 쓰세요 (README 참고).

## 0. GPU 확인 — 없으면 런타임을 GPU로 바꾸세요

In [ ]:
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('='*56)
    print('⚠️  GPU가 꺼져 있습니다. VLA 추론에는 GPU가 필요합니다.')
    print('   런타임 → 런타임 유형 변경 → T4 GPU → 모두 실행')
    print('='*56)

## 1. 라이브러리 설치 (약 1~2분)

In [ ]:
# transformers 는 huggingface_hub<1.0 을 요구 → 버전 고정(안 그러면 import 깨짐)
!pip install -q 'transformers>=4.57' peft accelerate 'huggingface_hub<1.0' opencv-python-headless matplotlib
print('설치 완료')

## 2. 사전 학습 산출물 — HuggingFace Hub에서 import
데이터 수집·GPU 학습을 건너뛰고, 미리 학습된 **웨이포인트 헤드**만 내려받습니다(약 10MB).

In [ ]:
from huggingface_hub import hf_hub_download
REPO = 'hoonsy/VLA_Simulation-pretrained'
HEAD_PT = hf_hub_download(repo_id=REPO, filename='vla_lora_head_fast.pt')
print('헤드 다운로드:', HEAD_PT)

## 3. 샘플 카메라 이미지 받기
실제 시뮬레이터에서 찍은 앞 카메라 프레임 6장(직선·좌커브·우커브 등).

In [ ]:
import os, urllib.request
RAW = 'https://raw.githubusercontent.com/SKKUAutoLab/VLA_Simulation/main/colab/samples'
NAMES = ['01_curve_left.jpg','02_curve_right.jpg','03_sharp_right.jpg',
         '04_lane0.jpg','05_lane1.jpg','06_curve_left2.jpg']
os.makedirs('samples', exist_ok=True)
for n in NAMES:
    urllib.request.urlretrieve(f'{RAW}/{n}', f'samples/{n}')
print('샘플', len(NAMES), '장 준비 완료')

## 4. 모델 정의 & 로딩
**비전 인코더**(Qwen3-VL-2B, 고정) + **웨이포인트 헤드**(우리가 학습한 부분).
헤드는 이미지 토큰(7×10 공간 격자)의 위치를 보존해 차선의 좌우 치우침을 읽습니다.

In [ ]:
import torch, torch.nn as nn
from transformers import Qwen3VLForConditionalGeneration, Qwen3VLProcessor
from PIL import Image as PILImage

BASE='Qwen/Qwen3-VL-2B-Instruct'; PIXELS=320*240; FEAT_DIM=2048; NTOK=70; DEV='cuda:0'

# --- 웨이포인트 헤드 (저장소 train_vla_lora.Head 와 동일) ---
class Head(nn.Module):
    def __init__(self, nout, ntok=NTOK):
        super().__init__()
        self.tok  = nn.Linear(FEAT_DIM, 64)          # 토큰별 2048→64
        self.pos  = nn.Parameter(torch.zeros(1, ntok, 64))  # 위치 임베딩(공간)
        self.proj = nn.Linear(ntok*64, 512)          # 전 토큰 평탄화(공간 보존)
        self.film = nn.Embedding(4, 1024)            # 차선 조건(FiLM)
        self.net  = nn.Sequential(nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512,256), nn.ReLU(), nn.Linear(256,nout))
    def forward(self, feat, lane):
        h = self.tok(feat) + self.pos
        h = self.proj(h.flatten(1))
        g, b = self.film(lane).chunk(2, dim=-1)
        h = h*(1+g) + b
        return self.net(h)

print('비전 인코더 로딩... (Qwen3-VL-2B, 최초 1회 다운로드 ~4GB)')
_m   = Qwen3VLForConditionalGeneration.from_pretrained(
         BASE, dtype=torch.bfloat16, device_map=DEV, attn_implementation='sdpa').eval()
proc = Qwen3VLProcessor.from_pretrained(BASE, min_pixels=PIXELS, max_pixels=PIXELS)
vis  = _m.model.visual.eval()

ck = torch.load(HEAD_PT, map_location=DEV)
head = Head(ck['nout']).float().to(DEV); head.load_state_dict(ck['state_dict']); head.eval()
WP_N, WP_SCALE = ck['wp_n'], ck['wp_scale']
print(f'로딩 완료 — 웨이포인트 {WP_N}개, 스케일 {WP_SCALE}m')

DUMMY = proc.apply_chat_template(
  [{'role':'user','content':[{'type':'image','image':PILImage.new('RGB',(8,8))},
                             {'type':'text','text':'x'}]}],
  tokenize=False, add_generation_prompt=True)

## 5. 추론 & 시각화 — 카메라 → 웨이포인트 → 조향
위=카메라 화면, 아래=모델이 예측한 주행 경로(위에서 본 그림).
빨간 삼각형=자동차, 파란 점=가야 할 웨이포인트. 조향 st는 -7(왼쪽)~+7(오른쪽).

In [ ]:
import cv2, numpy as np, math, glob
import matplotlib; import matplotlib.pyplot as plt
LOOKAHEAD, GAIN = 0.55, 13.0   # pure-pursuit (저장소 주행 노드와 동일)

@torch.inference_mode()
def predict(bgr, lane=0):
    pil = PILImage.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    inp = proc(text=[DUMMY], images=[pil], return_tensors='pt').to(DEV)
    out = vis(inp['pixel_values'].float(), grid_thw=inp['image_grid_thw'])
    feat = (out[0] if isinstance(out,(list,tuple)) else out).float().view(1, NTOK, FEAT_DIM)
    lane_t = torch.tensor([lane], dtype=torch.long, device=DEV)
    wp = head(feat, lane_t)[0].cpu().numpy() * WP_SCALE
    pts = [(wp[2*k], wp[2*k+1]) for k in range(WP_N)]      # (전방x, 횡y) m
    ex, ey = next(((a,b) for a,b in pts if a>=LOOKAHEAD), pts[-1])
    st = int(max(-7, min(7, round(-math.atan2(ey, max(0.3,ex))*GAIN))))
    return pts, st

files = sorted(glob.glob('samples/*.jpg'))
fig, axes = plt.subplots(2, len(files), figsize=(3.2*len(files), 6))
for j, ip in enumerate(files):
    bgr = cv2.imread(ip); pts, st = predict(bgr, lane=0)
    d = 'LEFT' if st<0 else ('RIGHT' if st>0 else 'STRAIGHT')
    axes[0,j].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); axes[0,j].axis('off')
    axes[0,j].set_title(f'steering st={st:+d}  {d}', fontsize=11)
    xs=[p[0] for p in pts]; ys=[p[1] for p in pts]
    axes[1,j].plot([0]+ys, [0]+xs, 'o-', color='tab:blue')
    axes[1,j].scatter([0],[0], c='red', marker='^', s=90)
    axes[1,j].set_xlim(2.5,-2.5); axes[1,j].set_ylim(-0.5, WP_SCALE)
    axes[1,j].set_aspect('equal'); axes[1,j].grid(alpha=0.3)
    axes[1,j].set_xlabel('lateral y (m)'); axes[1,j].set_ylabel('forward x (m)')
plt.tight_layout(); plt.show()
print('\n✅ 완료 — 커브 방향에 맞게 웨이포인트가 휘고 조향이 바뀌면 성공입니다.')

## (선택) 직접 확인해보기
본인 이미지를 `samples/` 에 올리고 위 셀을 다시 실행하거나, `lane` 을 1로 바꿔
(차선 조건 FiLM) 경로가 달라지는지 관찰해 보세요.

In [ ]:
# 예: 같은 이미지, 차선 조건 0 vs 1 비교
bgr = cv2.imread(sorted(glob.glob('samples/*.jpg'))[0])
for lane in (0,1):
    pts, st = predict(bgr, lane=lane)
    print(f'lane={lane}: 조향 st={st:+d}, 첫 웨이포인트=({pts[0][0]:.2f},{pts[0][1]:.2f})')